[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/B.%20Core%20Quay-Side%20Operations%20%28The%20Ship-to-Shore%20Interface%29/08.%20The%20Quay%20Crane%20Assignment%20Problem/P8-Tier-3_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---


# 8. The Quay Crane Assignment Problem
## Tier 3 — Differential Evolution

### Goal
This notebook applies Differential Evolution (DE), a population-based stochastic search method, to a quay crane assignment instance. Each candidate solution is represented by a real-valued matrix, decoded into a feasible assignment, and evaluated by its total vessel completion time. DE then improves the population using mutation, crossover, and selection.

### Inputs, outputs, and scope
- **Inputs:** Vessel workloads and crane-count bounds, crane productivity rates, an interference factor, and DE settings.
- **Output:** A feasible crane-to-vessel assignment, its vessel completion times, and diagnostic histories for fitness and population diversity.
- **Objective:** Minimize the sum of vessel completion times. This is not the makespan objective, which would minimize the time until the last vessel is complete.
- **Scope and assumptions:** Crane productivity is deterministic for each vessel–crane pair. Cranes are indivisible and can be assigned to at most one vessel. The decoder enforces each vessel's minimum and maximum crane counts. Interference is represented by a fixed multiplicative reduction for each additional crane assigned to the same vessel. Vessel sequencing, crane travel, spatial separation, setup time, and uncertainty are not modeled.

The example uses 5 vessels and 8 cranes, with a population of 40 individuals and 500 generations. The algorithm is stochastic: a run can produce different assignments unless the random-number generator is controlled. No claim of global optimality is made.

## Differential Evolution and the Assignment Model

A population is a collection of candidate encodings. Mutation combines three distinct population members to propose a new candidate; crossover mixes that proposal with the current candidate; selection keeps whichever decoded assignment has the lower objective value. These steps are repeated so that promising candidates can influence later search while stochastic variation continues to explore the encoding space.

Let \(V\) be the set of vessels and \(C\) the set of cranes. An assignment gives each vessel a set of cranes, with no crane assigned to more than one vessel. For vessel \(v\), let \(W_v\) be its workload, let \(r_{vc}\) be the productivity of crane \(c\) when serving vessel \(v\), and let \(n_v\) be the number of cranes assigned to it. The model's effective productivity and completion time are

$$
R_v = \sum_{c\ \text{assigned to }v} r_{vc}(1-\alpha)^{n_v-1},
\qquad
T_v = \frac{W_v}{R_v}.
$$

Here, \(\alpha\) is the interference factor. The same multiplicative penalty applies to each assigned crane, so the total effective productivity is the sum of the penalized individual rates. The objective used by `calculate_fitness` is

$$
\min \sum_{v\in V} T_v,
\qquad
\ell_v \le n_v \le u_v,
\qquad
\text{each crane is assigned to at most one vessel},
$$

where \(\ell_v\) and \(u_v\) are the vessel's minimum and maximum crane counts. In the code, these quantities are `workload`, `productivity_rates`, `interference_factor`, `min_cranes`, and `max_cranes`. The completion-time calculation is implemented by `calculate_completion_time_heuristic`; the aggregate objective is implemented by `calculate_fitness`.

The encoding is continuous, but assignments are discrete. Therefore, `decode_solution` is essential: it turns each real-valued matrix into an assignment that respects the crane-count bounds and crane exclusivity. This decoder is a deterministic heuristic, not an exact optimization of the assignment for a given encoding. DE searches the real-valued encoding space, and fitness is measured only after decoding.

## Step 1: Encoding, Decoding, and Fitness

An encoding is a matrix with one row per vessel and one column per crane. Its entries lie in \([0,1]\) and act as scores: among cranes that have not yet been assigned, higher-scoring cranes are considered first for a vessel. The decoder processes vessels in descending workload order. To preserve feasibility for vessels processed later, it reserves enough cranes to meet their minimum requirements before assigning optional cranes to the current vessel.

The encoding values also influence how many cranes a vessel receives. The decoder uses the mean score of the currently available cranes: a high mean favors the feasible maximum, a middle mean favors approximately half that maximum subject to the minimum, and a low mean favors the minimum. Thus, the encoding affects both the selected cranes and the preferred crane count. Feasibility requires that the sum of all minimum crane requirements does not exceed the available crane count.

In [ ]:
from dataclasses import dataclass
from typing import Dict, List, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


@dataclass
class Vessel:
    idx: int
    workload: int  # TEU
    min_cranes: int
    max_cranes: int
    name: str


# The vessel data define workload and the permitted crane-count interval.
vessels = [
    Vessel(0, 2000, 2, 4, "V1"),
    Vessel(1, 1500, 1, 3, "V2"),
    Vessel(2, 2500, 2, 5, "V3"),
    Vessel(3, 1000, 1, 2, "V4"),
    Vessel(4, 1800, 1, 3, "V5"),
]

# Rows correspond to vessels; columns correspond to cranes. Units are TEU/hour.
productivity_rates = np.array(
    [
        [25, 27, 24, 26, 23, 25, 26, 24],
        [23, 25, 26, 24, 22, 24, 25, 23],
        [26, 24, 25, 27, 25, 26, 24, 25],
        [24, 26, 23, 25, 26, 24, 23, 25],
        [25, 23, 26, 24, 25, 27, 24, 26],
    ]
)

# Each additional crane applies this fixed multiplicative interference penalty.
interference_factor = 0.08

# Differential Evolution settings for the concrete example.
POPULATION_SIZE = 40
MAX_GENERATIONS = 500
MUTATION_FACTOR = 0.7
CROSSOVER_RATE = 0.8


def encode_solution_random(
    vessels: Sequence[Vessel], productivity_rates: np.ndarray
) -> np.ndarray:
    """Create a random real-valued encoding with one score per vessel–crane pair."""
    return np.random.random((len(vessels), productivity_rates.shape[1]))


def decode_solution(
    encoding: np.ndarray,
    vessels: Sequence[Vessel],
    productivity_rates: np.ndarray,
) -> Dict[int, List[int]]:
    """Decode scores into an assignment satisfying crane-count and exclusivity rules."""
    if encoding.shape != (len(vessels), productivity_rates.shape[1]):
        raise ValueError("The encoding shape must match the vessel and crane counts.")
    if not np.all(np.isfinite(encoding)):
        raise ValueError("Encoding values must be finite.")
    if any(vessel.idx != index for index, vessel in enumerate(vessels)):
        raise ValueError("Vessel indices must match their positions in the vessel list.")

    crane_count = productivity_rates.shape[1]
    minimum_total = sum(vessel.min_cranes for vessel in vessels)
    if minimum_total > crane_count:
        raise ValueError("The minimum crane requirements exceed the available crane count.")
    if any(vessel.min_cranes < 0 or vessel.max_cranes < vessel.min_cranes for vessel in vessels):
        raise ValueError("Each vessel must have valid minimum and maximum crane bounds.")

    assignment: Dict[int, List[int]] = {}
    used_cranes = set()
    vessel_order = sorted(
        range(len(vessels)),
        key=lambda vessel_index: vessels[vessel_index].workload,
        reverse=True,
    )

    for position, vessel_index in enumerate(vessel_order):
        vessel = vessels[vessel_index]
        available_cranes = [
            crane_index
            for crane_index in range(crane_count)
            if crane_index not in used_cranes
        ]
        remaining_vessels = [vessels[index] for index in vessel_order[position + 1 :]]
        cranes_reserved_for_later = sum(
            later_vessel.min_cranes for later_vessel in remaining_vessels
        )

        # Reserving later minimum requirements prevents workload order from making
        # a valid instance infeasible for vessels that have not yet been processed.
        max_possible = min(
            vessel.max_cranes,
            len(available_cranes) - cranes_reserved_for_later,
        )
        if max_possible < vessel.min_cranes:
            raise ValueError("The decoder cannot satisfy all vessel crane-count bounds.")

        crane_scores = sorted(
            ((crane_index, encoding[vessel_index, crane_index])
             for crane_index in available_cranes),
            key=lambda item: item[1],
            reverse=True,
        )
        average_score = float(np.mean([score for _, score in crane_scores]))

        if average_score > 0.7:
            preferred_count = max_possible
        elif average_score > 0.4:
            preferred_count = max(vessel.min_cranes, max_possible // 2)
        else:
            preferred_count = vessel.min_cranes

        number_to_assign = min(max_possible, max(vessel.min_cranes, preferred_count))
        selected_cranes = [
            crane_index
            for crane_index, _ in crane_scores[:number_to_assign]
        ]
        assignment[vessel_index] = selected_cranes
        used_cranes.update(selected_cranes)

    return assignment


def calculate_completion_time_heuristic(
    vessel: Vessel,
    assigned_cranes: List[int],
    productivity_rates: np.ndarray,
    interference_factor: float,
) -> float:
    """Calculate a vessel's completion time in hours under the interference model."""
    if not assigned_cranes:
        return float("inf")
    if not 0 <= interference_factor < 1:
        raise ValueError("The interference factor must lie in [0, 1).")

    rates = productivity_rates[vessel.idx, assigned_cranes]
    crane_count = len(assigned_cranes)
    interference_penalty = (1 - interference_factor) ** (crane_count - 1)
    total_productivity = float(np.sum(rates * interference_penalty))
    if total_productivity <= 0:
        return float("inf")

    return vessel.workload / total_productivity


def calculate_fitness(
    assignment: Dict[int, List[int]],
    vessels: Sequence[Vessel],
    productivity_rates: np.ndarray,
    interference_factor: float,
) -> float:
    """Return the sum of vessel completion times; lower fitness is better."""
    expected_vessels = set(range(len(vessels)))
    if set(assignment) != expected_vessels:
        return float("inf")

    assigned_cranes = [
        crane_index
        for vessel_cranes in assignment.values()
        for crane_index in vessel_cranes
    ]
    if len(assigned_cranes) != len(set(assigned_cranes)):
        return float("inf")

    total_time = 0.0
    for vessel_index, cranes in assignment.items():
        vessel = vessels[vessel_index]
        if not vessel.min_cranes <= len(cranes) <= vessel.max_cranes:
            return float("inf")
        completion_time = calculate_completion_time_heuristic(
            vessel, cranes, productivity_rates, interference_factor
        )
        if not np.isfinite(completion_time):
            return float("inf")
        total_time += completion_time

    return total_time


print("Differential Evolution setup:")
print(f"Vessels: {len(vessels)}, Cranes: {productivity_rates.shape[1]}")
print(f"Population size: {POPULATION_SIZE}, Generations: {MAX_GENERATIONS}")
print(f"DE parameters: F={MUTATION_FACTOR}, CR={CROSSOVER_RATE}")
print("\nVessel details:")
for vessel in vessels:
    print(
        f"  {vessel.name}: {vessel.workload} TEU, "
        f"Cranes {vessel.min_cranes}-{vessel.max_cranes}"
    )

## Step 2: Evaluate Random Encodings

Before evolving a population, sample several random encodings and pass each through the same decoder and fitness function used by DE. This provides a baseline view of the objective values produced by the encoding–decoding pipeline. The sample is stochastic, so its histogram and summary statistics describe only the encodings generated in this run; they are not guaranteed bounds on the best achievable objective.

In [ ]:
random_fitnesses = []
for _ in range(10):
    encoding = encode_solution_random(vessels, productivity_rates)
    assignment = decode_solution(encoding, vessels, productivity_rates)
    fitness = calculate_fitness(
        assignment, vessels, productivity_rates, interference_factor
    )
    if np.isfinite(fitness):
        random_fitnesses.append(fitness)

if random_fitnesses:
    plt.figure(figsize=(8, 6))
    plt.hist(random_fitnesses, bins=10, alpha=0.7, edgecolor="black")
    mean_fitness = float(np.mean(random_fitnesses))
    plt.axvline(
        mean_fitness,
        color="red",
        linestyle="--",
        linewidth=2,
        label=f"Mean: {mean_fitness:.2f} h",
    )
    plt.xlabel("Total vessel completion time (hours)")
    plt.ylabel("Frequency")
    plt.title("Fitness Distribution of Random Encodings")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

    print(
        "Random-encoding fitness range: "
        f"{min(random_fitnesses):.2f}–{max(random_fitnesses):.2f} hours"
    )
    print(f"Average random-encoding fitness: {mean_fitness:.2f} hours")
else:
    print("No finite-fitness random encodings were generated.")

## Step 3: Differential Evolution Algorithm

The implementation uses the DE/rand/1 mutation strategy. For target individual \(x_i\), it selects three distinct population members \(x_{r_1},x_{r_2},x_{r_3}\), none of which is the target, and forms the mutant

$$
v_i = x_{r_1} + F(x_{r_2}-x_{r_3}).
$$

The mutant's entries are clipped to \([0,1]\), the encoding bounds. Binomial crossover creates a trial vector by choosing each entry from the mutant with probability \(CR\), while forcing at least one entry to come from the mutant. Selection is greedy: the trial replaces its target only if its decoded assignment has strictly lower fitness. Consequently, the best fitness in the population cannot increase from one generation to the next.

For a population of size \(P\), mutation requires at least four individuals: the target plus three distinct alternatives. The example and sensitivity settings satisfy this requirement. The algorithm records the best-so-far fitness and the mean pairwise Euclidean distance between encodings as a measure of population diversity.

In [ ]:
class DifferentialEvolution:
    def __init__(
        self,
        population_size: int,
        mutation_factor: float,
        crossover_rate: float,
        vessels: Sequence[Vessel],
        productivity_rates: np.ndarray,
        interference_factor: float,
    ):
        if population_size < 4:
            raise ValueError("DE/rand/1 requires a population size of at least four.")
        if mutation_factor < 0:
            raise ValueError("The mutation factor must be nonnegative.")
        if not 0 <= crossover_rate <= 1:
            raise ValueError("The crossover rate must lie in [0, 1].")

        self.population_size = population_size
        self.F = mutation_factor
        self.CR = crossover_rate
        self.vessels = vessels
        self.productivity_rates = productivity_rates
        self.interference_factor = interference_factor

        self.population = [
            encode_solution_random(vessels, productivity_rates)
            for _ in range(population_size)
        ]
        self.fitnesses = [
            calculate_fitness(
                decode_solution(individual, vessels, productivity_rates),
                vessels,
                productivity_rates,
                interference_factor,
            )
            for individual in self.population
        ]

        self.best_idx = int(np.argmin(self.fitnesses))
        self.best_fitness = self.fitnesses[self.best_idx]
        self.best_solution = self.population[self.best_idx].copy()
        self.fitness_history = [self.best_fitness]
        self.diversity_history = []

    def mutate(self, target_idx: int) -> np.ndarray:
        """Create a bounded mutant using the DE/rand/1 strategy."""
        candidate_indices = list(range(self.population_size))
        candidate_indices.remove(target_idx)
        r1, r2, r3 = np.random.choice(candidate_indices, 3, replace=False)

        mutant = (
            self.population[r1]
            + self.F * (self.population[r2] - self.population[r3])
        )
        return np.clip(mutant, 0.0, 1.0)

    def crossover(self, target: np.ndarray, mutant: np.ndarray) -> np.ndarray:
        """Apply binomial crossover, forcing at least one mutant entry into the trial."""
        trial = target.copy()
        forced_index = np.random.randint(target.size)

        for flat_index in range(target.size):
            if np.random.random() < self.CR or flat_index == forced_index:
                row, column = np.unravel_index(flat_index, target.shape)
                trial[row, column] = mutant[row, column]

        return trial

    def calculate_diversity(self) -> float:
        """Return the mean pairwise Euclidean distance in encoding space."""
        total_distance = 0.0
        pair_count = 0

        for first_index in range(self.population_size):
            for second_index in range(first_index + 1, self.population_size):
                distance = np.linalg.norm(
                    self.population[first_index] - self.population[second_index]
                )
                total_distance += float(distance)
                pair_count += 1

        return total_distance / pair_count if pair_count else 0.0

    def evolve(self, generations: int) -> None:
        """Run DE for the requested number of generations and record diagnostics."""
        if generations < 0:
            raise ValueError("The number of generations must be nonnegative.")

        print(f"Starting DE evolution for {generations} generations...")
        for generation in range(generations):
            new_population = []
            new_fitnesses = []

            for target_idx in range(self.population_size):
                mutant = self.mutate(target_idx)
                trial = self.crossover(self.population[target_idx], mutant)
                trial_assignment = decode_solution(
                    trial, self.vessels, self.productivity_rates
                )
                trial_fitness = calculate_fitness(
                    trial_assignment,
                    self.vessels,
                    self.productivity_rates,
                    self.interference_factor,
                )

                if trial_fitness < self.fitnesses[target_idx]:
                    new_population.append(trial)
                    new_fitnesses.append(trial_fitness)
                else:
                    new_population.append(self.population[target_idx])
                    new_fitnesses.append(self.fitnesses[target_idx])

            self.population = new_population
            self.fitnesses = new_fitnesses
            current_best_idx = int(np.argmin(self.fitnesses))
            current_best_fitness = self.fitnesses[current_best_idx]

            if current_best_fitness < self.best_fitness:
                self.best_fitness = current_best_fitness
                self.best_solution = self.population[current_best_idx].copy()
                self.best_idx = current_best_idx

            self.fitness_history.append(self.best_fitness)
            diversity = self.calculate_diversity()
            self.diversity_history.append(diversity)

            if (generation + 1) % 50 == 0:
                print(
                    f"Generation {generation + 1}: "
                    f"Best fitness = {self.best_fitness:.2f} hours, "
                    f"Diversity = {diversity:.4f}"
                )

        print(f"\nEvolution complete. Best fitness: {self.best_fitness:.2f} hours")


# Run the population search with the example settings.
de = DifferentialEvolution(
    POPULATION_SIZE,
    MUTATION_FACTOR,
    CROSSOVER_RATE,
    vessels,
    productivity_rates,
    interference_factor,
)
de.evolve(MAX_GENERATIONS)

best_assignment = decode_solution(
    de.best_solution, vessels, productivity_rates
)
best_assignment_fitness = calculate_fitness(
    best_assignment, vessels, productivity_rates, interference_factor
)

# These checks make the assignment constraints and reported objective explicit.
assert all(
    vessel.min_cranes <= len(best_assignment[index]) <= vessel.max_cranes
    for index, vessel in enumerate(vessels)
)
all_assigned_cranes = [
    crane for crane_list in best_assignment.values() for crane in crane_list
]
assert len(all_assigned_cranes) == len(set(all_assigned_cranes))
assert np.isclose(best_assignment_fitness, de.best_fitness)

print("\nBest decoded assignment:")
for vessel_idx, cranes in best_assignment.items():
    vessel = vessels[vessel_idx]
    completion_time = calculate_completion_time_heuristic(
        vessel, cranes, productivity_rates, interference_factor
    )
    crane_names = [f"C{crane_index + 1}" for crane_index in cranes]
    print(
        f"  {vessel.name}: {crane_names} ({len(cranes)} cranes) "
        f"-> {completion_time:.2f} hours"
    )
print(f"\nSum of vessel completion times: {best_assignment_fitness:.2f} hours")

## Step 4: Convergence Analysis and Visualization

The best-so-far fitness history records the smallest objective found up to each generation, while the diversity history records the population's mean pairwise distance after each generation. Fitness describes solution quality; diversity describes spread in the continuous encoding space. A decrease in encoding diversity does not by itself prove that all decoded assignments are identical, because different encodings can decode to the same assignment.

In [ ]:
generations = list(range(len(de.fitness_history)))
fitness_values = de.fitness_history
diversity_values = de.diversity_history

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# Best-so-far fitness should be nonincreasing under greedy selection.
axes[0, 0].plot(generations, fitness_values, color="blue", linewidth=2, alpha=0.8)
axes[0, 0].set_xlabel("Generation")
axes[0, 0].set_ylabel("Best fitness (hours)")
axes[0, 0].set_title("Best-So-Far Fitness")
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].axhline(
    y=de.best_fitness,
    color="red",
    linestyle="--",
    alpha=0.7,
    label=f"Best found: {de.best_fitness:.2f} h",
)
axes[0, 0].legend()

axes[0, 1].plot(
    generations[1:], diversity_values, color="green", linewidth=2, alpha=0.8
)
axes[0, 1].set_xlabel("Generation")
axes[0, 1].set_ylabel("Mean pairwise encoding distance")
axes[0, 1].set_title("Population Diversity")
axes[0, 1].grid(True, alpha=0.3)

improvement_by_generation = -np.diff(fitness_values)
axes[0, 2].plot(
    generations[1:],
    improvement_by_generation,
    color="red",
    linewidth=2,
    alpha=0.8,
)
axes[0, 2].set_xlabel("Generation")
axes[0, 2].set_ylabel("Decrease in best fitness (hours)")
axes[0, 2].set_title("Generation-by-Generation Improvement")
axes[0, 2].grid(True, alpha=0.3)

final_fitnesses = de.fitnesses
axes[1, 0].hist(final_fitnesses, bins=15, alpha=0.7, edgecolor="black")
axes[1, 0].axvline(
    de.best_fitness,
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"Best: {de.best_fitness:.2f} h",
)
axes[1, 0].axvline(
    np.mean(final_fitnesses),
    color="blue",
    linestyle="--",
    linewidth=2,
    label=f"Mean: {np.mean(final_fitnesses):.2f} h",
)
axes[1, 0].set_xlabel("Fitness (hours)")
axes[1, 0].set_ylabel("Number of individuals")
axes[1, 0].set_title("Final Population Fitness")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

vessel_names = [vessel.name for vessel in vessels]
crane_counts = [len(best_assignment[index]) for index in range(len(vessels))]
axes[1, 1].bar(
    vessel_names, crane_counts, color="skyblue", edgecolor="black", alpha=0.8
)
axes[1, 1].set_xlabel("Vessel")
axes[1, 1].set_ylabel("Number of assigned cranes")
axes[1, 1].set_title("Crane Counts in the Best Assignment")
axes[1, 1].grid(True, axis="y", alpha=0.3)

completion_times = [
    calculate_completion_time_heuristic(
        vessels[index],
        best_assignment[index],
        productivity_rates,
        interference_factor,
    )
    for index in range(len(vessels))
]
axes[1, 2].bar(
    vessel_names, completion_times, color="orange", edgecolor="black", alpha=0.8
)
axes[1, 2].set_xlabel("Vessel")
axes[1, 2].set_ylabel("Completion time (hours)")
axes[1, 2].set_title("Completion Time by Vessel")
axes[1, 2].grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

initial_fitness = fitness_values[0]
final_fitness = de.best_fitness
improvement_percent = (initial_fitness - final_fitness) / initial_fitness * 100
convergence_generation = next(
    index
    for index, fitness in enumerate(fitness_values)
    if np.isclose(fitness, final_fitness, rtol=1e-9, atol=1e-12)
)

print("\nDifferential Evolution performance summary:")
print(f"  Initial best fitness: {initial_fitness:.2f} hours")
print(f"  Final best fitness: {final_fitness:.2f} hours")
print(f"  Improvement from initial best: {improvement_percent:.1f}%")
print(f"  First generation attaining the final best: {convergence_generation}")
if diversity_values:
    print(f"  Final population diversity: {diversity_values[-1]:.4f}")
print(f"  Final population fitness standard deviation: {np.std(final_fitnesses):.2f}")

# Compare best-so-far fitness at three generation intervals.
first_boundary = MAX_GENERATIONS // 3
second_boundary = 2 * MAX_GENERATIONS // 3
phase_ranges = [
    ("Early", 0, first_boundary),
    ("Middle", first_boundary, second_boundary),
    ("Late", second_boundary, MAX_GENERATIONS),
]
print("\nBest-fitness decrease within each phase:")
for phase_name, start, end in phase_ranges:
    phase_values = fitness_values[start : end + 1]
    phase_start = phase_values[0]
    phase_best = min(phase_values)
    phase_decrease = (phase_start - phase_best) / phase_start * 100
    print(
        f"  {phase_name} (generations {start}-{end}): "
        f"{phase_decrease:.1f}%"
    )

## Step 5: Parameter Sensitivity Analysis

The mutation factor \(F\), crossover rate \(CR\), and population size affect how DE proposes and retains candidate encodings. This experiment varies each setting while keeping the vessel instance and fitness model unchanged. Each tested combination is a stochastic run, so the results describe those particular runs rather than a guaranteed ranking that holds across random initializations.

In [ ]:
def test_de_parameters(
    pop_sizes: List[int],
    f_values: List[float],
    cr_values: List[float],
    vessels: Sequence[Vessel],
    productivity_rates: np.ndarray,
    interference_factor: float,
    generations: int = 100,
) -> pd.DataFrame:
    """Run the specified DE parameter combinations and collect their outcomes."""
    results = []

    for population_size in pop_sizes:
        for mutation_factor in f_values:
            for crossover_rate in cr_values:
                print(
                    f"Testing: Population={population_size}, "
                    f"F={mutation_factor}, CR={crossover_rate}"
                )
                test_de = DifferentialEvolution(
                    population_size,
                    mutation_factor,
                    crossover_rate,
                    vessels,
                    productivity_rates,
                    interference_factor,
                )
                test_de.evolve(generations)

                initial = test_de.fitness_history[0]
                final = test_de.best_fitness
                convergence_generation = next(
                    index
                    for index, fitness in enumerate(test_de.fitness_history)
                    if np.isclose(fitness, final, rtol=1e-9, atol=1e-12)
                )
                results.append(
                    {
                        "Population_Size": population_size,
                        "F": mutation_factor,
                        "CR": crossover_rate,
                        "Final_Fitness": final,
                        "Initial_Fitness": initial,
                        "Improvement": (initial - final) / initial * 100,
                        "Convergence_Gen": convergence_generation,
                    }
                )

    return pd.DataFrame(results)


param_results = test_de_parameters(
    pop_sizes=[20, 40, 60],
    f_values=[0.5, 0.7, 0.9],
    cr_values=[0.6, 0.8, 1.0],
    vessels=vessels,
    productivity_rates=productivity_rates,
    interference_factor=interference_factor,
    generations=50,
)

print("Parameter sensitivity results, ordered by final fitness:")
print(param_results.sort_values("Final_Fitness").head(10).to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

population_groups = param_results.groupby("Population_Size")["Final_Fitness"].mean()
axes[0].bar(
    population_groups.index.astype(str),
    population_groups.values,
    color="lightblue",
    edgecolor="black",
)
axes[0].set_xlabel("Population size")
axes[0].set_ylabel("Mean final fitness (hours)")
axes[0].set_title("Effect of Population Size")
axes[0].grid(True, axis="y", alpha=0.3)

mutation_groups = param_results.groupby("F")["Final_Fitness"].mean()
axes[1].bar(
    mutation_groups.index.astype(str),
    mutation_groups.values,
    color="lightgreen",
    edgecolor="black",
)
axes[1].set_xlabel("Mutation factor (F)")
axes[1].set_ylabel("Mean final fitness (hours)")
axes[1].set_title("Effect of Mutation Factor")
axes[1].grid(True, axis="y", alpha=0.3)

crossover_groups = param_results.groupby("CR")["Final_Fitness"].mean()
axes[2].bar(
    crossover_groups.index.astype(str),
    crossover_groups.values,
    color="lightcoral",
    edgecolor="black",
)
axes[2].set_xlabel("Crossover rate (CR)")
axes[2].set_ylabel("Mean final fitness (hours)")
axes[2].set_title("Effect of Crossover Rate")
axes[2].grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

best_parameter_row = param_results.loc[param_results["Final_Fitness"].idxmin()]
print("\nBest observed parameter combination in these runs:")
print(f"  Population size: {int(best_parameter_row['Population_Size'])}")
print(f"  F: {best_parameter_row['F']}")
print(f"  CR: {best_parameter_row['CR']}")
print(f"  Final fitness: {best_parameter_row['Final_Fitness']:.2f} hours")
print(f"  Improvement: {best_parameter_row['Improvement']:.1f}%")

## Interpreting the Results

The plots and tables provide several complementary views of the search:

- **Best-so-far fitness** shows the quality of the best decoded assignment found up to each generation. Under the implemented greedy selection rule, this history is nonincreasing.
- **Population diversity** measures distances between continuous encodings. Low diversity can indicate that the population has clustered, but it does not establish that the search has found a global optimum.
- **Final fitness distribution** shows how candidate quality varies within the final population; the best individual is not necessarily representative of the population mean.
- **Assignment and completion-time plots** describe the best decoded assignment found in this run. The total objective is the sum of the vessel completion times shown in the breakdown.
- **Parameter sensitivity summaries** average outcomes over the tested combinations in this experiment. Because the algorithm is stochastic and each combination is run once here, the observed best combination is not evidence that it is universally best.

Differential Evolution combines population-based exploration with selection that favors lower objective values. Mutation introduces variation, crossover transfers parts of a mutant into a trial encoding, and selection preserves a trial only when its decoded assignment improves fitness. These mechanisms can find useful solutions, but they do not guarantee escape from every local optimum or guarantee a globally optimal assignment. Results depend on the decoder, objective model, parameter settings, and stochastic run.